<a href="https://colab.research.google.com/github/sharnitha567/Machine-Learning/blob/main/Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [176]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Load Dataset

In [177]:
student = pd.read_csv("/content/student_performance_updated_1000.csv")

print("Dataset:")
print(student.head())

print("\nDataset Shape:", student.shape)
print("\nNo of Rows:", student.shape[0])
print("No of Columns:", student.shape[1])

Dataset:
   StudentID     Name  Gender  AttendanceRate  StudyHoursPerWeek  \
0        1.0     John    Male            85.0               15.0   
1        2.0    Sarah  Female            90.0               20.0   
2        3.0     Alex    Male            78.0               10.0   
3        4.0  Michael    Male            92.0               25.0   
4        5.0     Emma  Female             NaN               18.0   

   PreviousGrade  ExtracurricularActivities ParentalSupport  FinalGrade  \
0           78.0                        1.0            High        80.0   
1           85.0                        2.0          Medium        87.0   
2           65.0                        0.0             Low        68.0   
3           90.0                        3.0            High        92.0   
4           82.0                        2.0          Medium        85.0   

   Study Hours  Attendance (%) Online Classes Taken  
0          4.8            59.0                False  
1          2.2         

Check Dataset Information

In [178]:
print("\nDataset Information:")
print(student.info())

print("\nMissing Values:")
print(student.isnull().sum())


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StudentID                  960 non-null    float64
 1   Name                       966 non-null    object 
 2   Gender                     952 non-null    object 
 3   AttendanceRate             960 non-null    float64
 4   StudyHoursPerWeek          950 non-null    float64
 5   PreviousGrade              967 non-null    float64
 6   ExtracurricularActivities  957 non-null    float64
 7   ParentalSupport            978 non-null    object 
 8   FinalGrade                 960 non-null    float64
 9   Study Hours                976 non-null    float64
 10  Attendance (%)             959 non-null    float64
 11  Online Classes Taken       975 non-null    object 
dtypes: float64(8), object(4)
memory usage: 93.9+ KB
None

Missing Values:
StudentID    

Create Classification Target

In [179]:
student["Grade_Category"] = np.where(
    student["FinalGrade"] >= 75,
    "Pass",
    "Fail"
)

print("\nGrade Category:")
print(student["Grade_Category"].value_counts())


Grade Category:
Grade_Category
Pass    667
Fail    333
Name: count, dtype: int64


Remove Unnecessary Columns

In [180]:
student = student.drop(
    ["StudentID", "Name"],
    axis=1
)

print("\nColumns after removing StudentID and Name:")
print(student.columns)


Columns after removing StudentID and Name:
Index(['Gender', 'AttendanceRate', 'StudyHoursPerWeek', 'PreviousGrade',
       'ExtracurricularActivities', 'ParentalSupport', 'FinalGrade',
       'Study Hours', 'Attendance (%)', 'Online Classes Taken',
       'Grade_Category'],
      dtype='object')


In [181]:
label_encoder = LabelEncoder()

categorical_columns = [
    "Gender",
    "ParentalSupport",
    "Online Classes Taken"
]

for column in categorical_columns:
    student[column] = label_encoder.fit_transform(
        student[column].astype(str)
    )

# --- NEW: Handle numerical missing values in 'student' dataframe ---
numeric_columns = student.select_dtypes(include=np.number).columns

for col in numeric_columns:
    # Impute only if there are NaNs in the column
    if student[col].isnull().any():
        student[col] = student[col].fillna(student[col].median())

print("\nEncoded and Imputed Dataset (first 5 rows):")
print(student.head())

print("\nMissing values in student after encoding and imputation:")
print(student.isnull().sum())


Encoded and Imputed Dataset (first 5 rows):
   Gender  AttendanceRate  StudyHoursPerWeek  PreviousGrade  \
0       1            85.0               15.0           78.0   
1       0            90.0               20.0           85.0   
2       1            78.0               10.0           65.0   
3       1            92.0               25.0           90.0   
4       0            88.0               18.0           82.0   

   ExtracurricularActivities  ParentalSupport  FinalGrade  Study Hours  \
0                        1.0                0        80.0          4.8   
1                        2.0                2        87.0          2.2   
2                        0.0                1        68.0          4.6   
3                        3.0                0        92.0          2.9   
4                        2.0                2        85.0          4.1   

   Attendance (%)  Online Classes Taken Grade_Category  
0            59.0                     0           Pass  
1            70.0

Separate Features and Target

In [182]:
X = student.drop(
    ["FinalGrade", "Grade_Category"],
    axis=1
)

y = student["Grade_Category"]

print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())


Features:
   Gender  AttendanceRate  StudyHoursPerWeek  PreviousGrade  \
0       1            85.0               15.0           78.0   
1       0            90.0               20.0           85.0   
2       1            78.0               10.0           65.0   
3       1            92.0               25.0           90.0   
4       0            88.0               18.0           82.0   

   ExtracurricularActivities  ParentalSupport  Study Hours  Attendance (%)  \
0                        1.0                0          4.8            59.0   
1                        2.0                2          2.2            70.0   
2                        0.0                1          4.6            92.0   
3                        3.0                0          2.9            96.0   
4                        2.0                2          4.1            97.0   

   Online Classes Taken  
0                     0  
1                     1  
2                     0  
3                     0  
4          

Train/Test Split

In [183]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (800, 9)
X_test : (200, 9)
y_train: (800,)
y_test : (200,)


Feature Scaling

In [184]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaled Training Data:")
print(X_train_scaled[:5])

print("\nScaled Testing Data:")
print(X_test_scaled[:5])


Scaled Training Data:
[[ 2.39122644 -2.16933325 -0.43197874 -1.81989841 -0.48843286 -1.15773369
   0.54079318 -1.21741948  2.71135011]
 [-1.04938714 -2.16933325 -0.10222398 -0.08118268 -0.48843286 -1.15773369
  -1.21203091  0.32328159 -1.01559162]
 [ 0.67091965 -0.49860728  0.39240816 -0.79712445 -1.44849746 -1.15773369
   0.47819232 -0.20799464 -1.01559162]
 [ 0.67091965  1.31134585 -0.92661088  0.83931388  0.47163173 -1.15773369
   0.54079318 -0.26112227  0.84787924]
 [-1.04938714 -0.08092579  0.0626534   0.4302043   1.43169633 -1.15773369
  -1.52503521  0.42953683 -1.01559162]]

Scaled Testing Data:
[[-1.04938714  1.31134585  0.39240816  0.73703648 -0.48843286 -0.01429301
  -0.5860223   0.5889197  -1.01559162]
 [-1.04938714  1.31134585  1.21679507 -0.08118268 -0.48843286 -1.15773369
   0.85379748  0.85455781 -1.01559162]
 [-1.04938714  1.31134585  2.04118197 -0.08118268 -0.48843286 -0.01429301
   0.0399863  -0.04861177 -1.01559162]
 [ 0.67091965  0.3367557   0.72216292 -1.30851143 

In [185]:
# Check missing values
print("Missing values in X:")
print(X.isnull().sum())

print("\nMissing values in y:")
print(pd.Series(y).isnull().sum())

Missing values in X:
Gender                       0
AttendanceRate               0
StudyHoursPerWeek            0
PreviousGrade                0
ExtracurricularActivities    0
ParentalSupport              0
Study Hours                  0
Attendance (%)               0
Online Classes Taken         0
dtype: int64

Missing values in y:
0


In [186]:

svm_model = SVC(
    kernel="rbf",
    C=1.0,
    random_state=42,
    class_weight="balanced"
)

svm_model.fit(
    X_train_scaled,
    y_train
)

print("\nSVM model trained successfully.")


SVM model trained successfully.


Create and Train SVM

In [187]:
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    random_state=42,
    class_weight='balanced' # Added to handle class imbalance
)

svm_model.fit(X_train_scaled, y_train)

print("SVM model trained successfully with class_weight='balanced'.")

SVM model trained successfully with class_weight='balanced'.


Prediction

In [188]:
y_pred = svm_model.predict(X_test_scaled)

print("\nPredicted Values:")
print(y_pred[:20])

print("\nActual Values:")
print(y_test.values[:20])


Predicted Values:
['Fail' 'Pass' 'Pass' 'Fail' 'Pass' 'Pass' 'Fail' 'Fail' 'Fail' 'Pass'
 'Fail' 'Pass' 'Pass' 'Fail' 'Fail' 'Fail' 'Pass' 'Pass' 'Fail' 'Fail']

Actual Values:
['Pass' 'Pass' 'Pass' 'Pass' 'Fail' 'Pass' 'Pass' 'Fail' 'Fail' 'Pass'
 'Pass' 'Fail' 'Fail' 'Fail' 'Pass' 'Pass' 'Pass' 'Fail' 'Pass' 'Fail']


Baseline Accuracy

In [189]:
baseline_accuracy = y_test.value_counts().max() / len(y_test)

print("Baseline Accuracy:", baseline_accuracy)
print("Baseline Accuracy (%):", baseline_accuracy * 100)

Baseline Accuracy: 0.665
Baseline Accuracy (%): 66.5


SVM Accuracy

In [190]:
accuracy = accuracy_score(y_test, y_pred)

print("SVM Accuracy:", accuracy)
print("SVM Accuracy (%):", accuracy * 100)

SVM Accuracy: 0.495
SVM Accuracy (%): 49.5


Confusion Matrix

In [191]:
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[34 33]
 [68 65]]


Classification Report

In [192]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

        Fail       0.33      0.51      0.40        67
        Pass       0.66      0.49      0.56       133

    accuracy                           0.49       200
   macro avg       0.50      0.50      0.48       200
weighted avg       0.55      0.49      0.51       200



In [193]:
print("Training class distribution:")
print(y_train.value_counts())

print("\nTesting class distribution:")
print(y_test.value_counts())

Training class distribution:
Grade_Category
Pass    534
Fail    266
Name: count, dtype: int64

Testing class distribution:
Grade_Category
Pass    133
Fail     67
Name: count, dtype: int64


In [194]:
kernels = [
    "linear",
    "poly",
    "rbf",
    "sigmoid"
]

print("\nAccuracy for Different Kernels:")

for kernel in kernels:

    model = SVC(
        kernel=kernel,
        C=1.0,
        random_state=42,
        class_weight="balanced"
    )

    model.fit(
        X_train_scaled,
        y_train
    )

    prediction = model.predict(
        X_test_scaled
    )

    kernel_accuracy = accuracy_score(
        y_test,
        prediction
    )

    print(
        f"{kernel} kernel accuracy: "
        f"{kernel_accuracy:.4f}"
    )


Accuracy for Different Kernels:
linear kernel accuracy: 0.4950
poly kernel accuracy: 0.5650
rbf kernel accuracy: 0.4950
sigmoid kernel accuracy: 0.4250
